# MyFirstLLM v2 — 一个真正能跑的 Decoder-only mini-GPT

> 修复了 v1 的所有致命问题：Causal Mask、Decoder-only 架构、TinyShakespeare 数据集、GPU 训练、train/val split、loss 曲线、模型保存。

## 🎯 本 Notebook 的目标

在 [TinyShakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) 数据集上，从零搭建并训练一个 **字符级 Decoder-only GPT**，最终能生成莎士比亚风格的英文文本。

## 🔄 相对 v1 的关键改进

| # | 问题 (v1) | 解决方案 (v2) |
|---|-----------|--------------|
| 1 | ❌ MultiHeadAttention 没有 causal mask，模型偷看未来 | ✅ `register_buffer` 注册下三角 mask，softmax 前 masked_fill |
| 2 | ❌ Encoder + Decoder 架构错配（自回归 LM 用 seq2seq） | ✅ 改为标准 Decoder-only（GPT-2 / nanoGPT 同款） |
| 3 | ❌ 6 句话训练，严重过拟合，loss=0.01 但生成是乱码 | ✅ 1MB TinyShakespeare + 9:1 train/val split |
| 4 | ❌ 全程 CPU，浪费 Colab 的 T4 GPU | ✅ 自动检测设备并 `.to(device)` |
| 5 | ❌ 无 weight tying、无 grad clipping、无 LR schedule | ✅ 全部加上（GPT-2 标配） |
| 6 | ❌ `try/except` 掩盖 NameError，cell 顺序混乱 | ✅ 按"导入 → 数据 → 模型 → 训练 → 推理"严格顺序 |
| 7 | ❌ 没有 loss 曲线、没有保存模型 | ✅ 记录 train/val loss 并 plot；训练后 `torch.save` |

## 🔗 关联笔记

- [[搭建一个Transformer]]
- [[为什么需要Causal Mask]]
- [[如何从Transformer扩展到GPT]]
- [[搭建Transformer的常见坑]]
- [[MyFirstLLM_review]]

## 1. 环境准备

固定随机种子，自动选择 GPU/CPU。在 Colab 中请先 `Runtime → Change runtime type → T4 GPU`。

In [ ]:
import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
torch.cuda.manual_seed_all(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 数据集：TinyShakespeare

约 1MB 的莎士比亚剧本字符级语料。65 个唯一字符（含大小写字母、标点、空白）。

> **为什么换数据集？** v1 用 6 句话训练，模型只能"背诵"无法"泛化"。1MB 量级的数据是字符级 LM 的最低门槛（参考 Karpathy 的 char-rnn）。

In [ ]:
DATA_PATH = 'tinyshakespeare.txt'
DATA_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

if not os.path.exists(DATA_PATH):
    import urllib.request
    print(f"Downloading from {DATA_URL} ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total chars: {len(text):,}")
print(f"First 200 chars:\n{text[:200]}")

## 3. 字符级 Tokenizer + Train/Val Split

最简单的 tokenizer：每个字符一个 ID。生产环境用 BPE/SentencePiece，但教学场景字符级最直观。

**关键改进**：90/10 切出验证集，用来检测过拟合（v1 没有验证集，loss 降到 0.01 是假象）。

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    return ''.join(itos[i] for i in ids)

print(f"Vocab size: {vocab_size}")
print(f"Vocab: {''.join(chars)!r}")

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}")

## 4. 模型超参数

> 这个配置在 Colab T4 GPU 上约 5-10 分钟训练 5000 步，最终 val loss ≈ 1.5 左右。
> 想训练更大模型，把 `n_layer / n_embd / block_size` 调大即可。

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = vocab_size
    block_size: int = 256       # 上下文窗口
    n_layer:    int = 6         # Transformer Block 层数
    n_head:     int = 6         # 注意力头数
    n_embd:     int = 384       # embedding 维度（必须能被 n_head 整除）
    dropout:    float = 0.2

@dataclass
class TrainConfig:
    batch_size:    int = 64
    max_iters:     int = 5000
    eval_interval: int = 500
    eval_iters:    int = 200
    learning_rate: float = 3e-4
    warmup_iters:  int = 100
    grad_clip:     float = 1.0

cfg   = GPTConfig()
tcfg  = TrainConfig()
print(cfg)
print(tcfg)

In [ ]:
def get_batch(split: str):
    """随机抽一个 batch；x: 当前 token, y: 下一个 token (右移一位)。"""
    src = train_data if split == 'train' else val_data
    ix = torch.randint(len(src) - cfg.block_size, (tcfg.batch_size,))
    x = torch.stack([src[i:i+cfg.block_size]      for i in ix])
    y = torch.stack([src[i+1:i+cfg.block_size+1]  for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(f"x shape: {xb.shape}, y shape: {yb.shape}")
print(f"Sample x[0][:30]: {decode(xb[0][:30].tolist())!r}")
print(f"Sample y[0][:30]: {decode(yb[0][:30].tolist())!r}  ← 右移一位")

## 5. 模型组件

### 5.1 CausalSelfAttention（v1 最致命的 bug 在这里修复）

**v1 的问题**：直接对 `Q @ K^T` 做 softmax，没有 mask → 模型可以看到未来 token → 训练时作弊 → 推理时崩溃。

**v2 的修复**：
1. `register_buffer` 注册下三角 mask（不参与梯度）
2. softmax 之前 `masked_fill(mask == 0, -inf)` → softmax 后未来位置权重为 0

> 详见笔记 [[为什么需要Causal Mask]]

In [ ]:
class CausalSelfAttention(nn.Module):
    """带 Causal Mask 的多头自注意力。"""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0, "n_embd 必须能被 n_head 整除"
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head

        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                 .view(1, 1, cfg.block_size, cfg.block_size),
        )

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

### 5.2 MLP（Feed-Forward Network）

经典的两层 MLP，中间维度放大 4 倍。激活函数用 GELU（GPT-2 标准）。

In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

### 5.3 Block（Pre-LN Transformer Block）

```
x = x + Attention(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```

> Pre-LN（先 LN 再 sublayer）比原版论文的 Post-LN 训练更稳定，是 GPT-2 之后的主流。

In [ ]:
class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp  = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### 5.4 GPT 主体（Decoder-only）

```
idx → token_emb + pos_emb → Block × N → ln_f → lm_head → logits
```

**关键工程细节**：

- **Weight tying**: `lm_head.weight = wte.weight`，输入 embedding 和输出投影共享权重，节省参数 + 训练更稳。
- **Pos embedding 用可学习的 `nn.Embedding`**（GPT-2 同款），比 sin/cos 更灵活。
- **forward 同时支持训练（带 targets）和推理（不带 targets）**。
- **`generate` 内置在模型里**，标准做法。

> 详见笔记 [[如何从Transformer扩展到GPT]]

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        self.wte  = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe  = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f   = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self) -> int:
        n = sum(p.numel() for p in self.parameters())
        n -= self.wpe.weight.numel()
        return n

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"序列长度 {T} 超过 block_size {self.cfg.block_size}"

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.wte(idx)              # (B, T, C)
        pos_emb = self.wpe(pos)              # (T, C)
        x = self.drop(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)             # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0, top_k: int | None = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

## 6. 实例化 + 一次前向 sanity check

实例化模型，做一次前向，验证：
- 参数量是否合理
- 输入输出 shape 是否对
- 初始 loss 是否接近 `-ln(1/vocab_size)`（≈ 4.17 对 65 个字符）—— 如果远偏离这个值，说明初始化有问题

In [ ]:
model = GPT(cfg).to(device)
print(f"Model parameters (excl. pos_emb): {model.num_params() / 1e6:.2f}M")

xb, yb = get_batch('train')
logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Initial loss: {loss.item():.4f}  (理论值 -ln(1/{vocab_size}) = {math.log(vocab_size):.4f})")

## 7. 训练循环

包含的工程最佳实践：

- **`@torch.no_grad()` + `model.eval()/train()` 切换** 评估 val loss
- **梯度裁剪** `clip_grad_norm_(max_norm=1.0)` 防止梯度爆炸
- **Linear warmup + cosine decay** 学习率调度
- **记录 train/val loss 曲线** 观察过拟合
- **进度日志** 每 500 步打印一次

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ('train', 'val'):
        losses = torch.zeros(tcfg.eval_iters)
        for k in range(tcfg.eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def get_lr(it: int) -> float:
    if it < tcfg.warmup_iters:
        return tcfg.learning_rate * (it + 1) / tcfg.warmup_iters
    progress = (it - tcfg.warmup_iters) / max(1, tcfg.max_iters - tcfg.warmup_iters)
    return tcfg.learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))

optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg.learning_rate, betas=(0.9, 0.95))

history = {'iter': [], 'train': [], 'val': []}
t0 = time.time()
for it in range(tcfg.max_iters + 1):

    lr = get_lr(it)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    if it % tcfg.eval_interval == 0 or it == tcfg.max_iters:
        losses = estimate_loss()
        history['iter'].append(it)
        history['train'].append(losses['train'])
        history['val'].append(losses['val'])
        elapsed = time.time() - t0
        print(f"iter {it:5d} | lr {lr:.2e} | train {losses['train']:.4f} | val {losses['val']:.4f} | {elapsed:6.1f}s")

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
    optimizer.step()

print(f"\nTotal training time: {(time.time() - t0)/60:.2f} min")

## 8. Loss 曲线

如果 train loss 持续下降但 val loss 触底反弹 → 过拟合，需要：增大 dropout / 减小模型 / 增加数据。

健康的曲线：train 和 val 同步下降，两者 gap 不超过 0.2。

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['iter'], history['train'], label='train', marker='o')
plt.plot(history['iter'], history['val'],   label='val',   marker='s')
plt.xlabel('Iteration')
plt.ylabel('Loss (cross entropy)')
plt.title('Training & Validation Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=120)
plt.show()

## 9. 推理：生成莎士比亚风格文本

对比三种采样策略：

- **temperature=0.5 + top_k=10**：保守，输出稳定但可能重复
- **temperature=0.8 + top_k=40**：平衡（推荐默认）
- **temperature=1.2**（无 top_k）：发散，更有创意但容易出错

In [ ]:
def sample(prompt: str, max_new_tokens: int = 300, **gen_kwargs) -> str:
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=max_new_tokens, **gen_kwargs)
    return decode(out[0].tolist())

prompt = "ROMEO:"

print("=" * 60)
print("temperature=0.5, top_k=10  (保守)")
print("=" * 60)
print(sample(prompt, max_new_tokens=300, temperature=0.5, top_k=10))

print("\n" + "=" * 60)
print("temperature=0.8, top_k=40  (推荐)")
print("=" * 60)
print(sample(prompt, max_new_tokens=300, temperature=0.8, top_k=40))

print("\n" + "=" * 60)
print("temperature=1.2  (发散)")
print("=" * 60)
print(sample(prompt, max_new_tokens=300, temperature=1.2))

## 10. 保存模型

保存 `state_dict` + 配置 + 字符表，方便以后断点续训或部署。

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': cfg.__dict__,
    'stoi': stoi,
    'itos': itos,
    'history': history,
}
torch.save(checkpoint, 'mygpt.pt')
print(f"Saved to mygpt.pt  (size: {os.path.getsize('mygpt.pt') / 1e6:.1f} MB)")

## 11. 总结与下一步

### ✅ 已完成
- 修复 v1 的 7 个核心问题
- 训出第一个能生成莎士比亚风格英文的 mini-GPT
- 完整记录 train/val loss 曲线 + 模型 checkpoint

### 🚀 下一步实验方向

1. **Scale up**：把 `n_layer=6 → 12`, `n_embd=384 → 768`，看 val loss 能下降到多少（验证 [[概率模型如何产生智能]] 中的 emergence）
2. **换 BPE tokenizer**：用 `tiktoken` 替换字符级 tokenizer，处理更长上下文
3. **Flash Attention**：用 `F.scaled_dot_product_attention` 替换手写 attention，速度提升 2-4 倍
4. **加 RoPE**：把绝对位置编码换成 RoPE，支持外推到更长序列
5. **微调实验**：用 LoRA 在小数据集上微调一个开源模型（HuggingFace `transformers` + `peft`）

### 📚 参考资源
- Karpathy [Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY)
- [nanoGPT repo](https://github.com/karpathy/nanoGPT)
- [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)